# AgroguardAI-LLM Training

Fine-tune Mistral-7B-Instruct-v0.3 on the AgroguardAI-LLM dataset (500+ QA pairs, 25 crops, 12 dialects) using QLoRA.
Runs on a free T4 GPU (16GB VRAM) in ~90 minutes for 452 training entries, 2 epochs.

**Before you start:** Mount Google Drive to save your trained adapter and enter your Hugging Face token.


In [ ]:
# @title 1. Install dependencies
!pip install -qU transformers peft accelerate bitsandbytes trl datasets huggingface_hub
!pip install -qU xformers --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# @title 2. Mount Google Drive (saves adapter here)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/agroguardai'
!mkdir -p {DRIVE_PATH}

In [ ]:
# @title 3. Clone dataset from GitHub
!git clone https://github.com/agroguardaiaOS/agroguardai-llm.git /content/agroguardai-llm
!ls /content/agroguardai-llm/data/

In [ ]:
# @title 4. Preprocess the dataset
%cd /content/agroguardai-llm
# Load from pre-split train/test files
!echo 'Training entries:'
!python -c "import json; d=json.load(open('data/processed/train.json')); print(len(d))"
!echo 'Test entries:'
!python -c "import json; d=json.load(open('data/processed/test.json')); print(len(d))"
!echo '---'
# Tokenize train + test sets
!python src/preprocess.py \
    --data data/processed/train.json \
    --output data/processed \
    --val-data data/processed/test.json \
    --seed 42

In [ ]:
# @title 5. Run training (QLoRA on Mistral-7B, 2 epochs)
# ~90 min on T4 GPU with 452 training entries, 2 epochs
# Keeps same hyperparameters (r=16, alpha=32, 4-bit NF4)

HF_TOKEN = ""  # @param {type:"string"}
# Paste your Hugging Face token to push the adapter to the Hub

import os, yaml
from pathlib import Path

cfg_path = '/content/agroguardai-llm/config/lora_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Override epochs to 2 for this training run
cfg['training']['num_epochs'] = 2

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub")
    cfg['output']['hub_model_id'] = 'agroguardaiaOS/agroguardai-llm-lora'
else:
    cfg['output']['hub_model_id'] = ''
    print("HF_TOKEN empty — adapter saved locally only")

# Write updated config
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f)

# Run training
!python src/train.py --config config/lora_config.yaml --data data/processed/

In [ ]:
# @title 6. Save adapter to Google Drive
!cp -r /content/agroguardai-llm/models/agroguardai-lora-adapter {DRIVE_PATH}/
!echo 'Adapter saved to Google Drive:'
!ls -lh {DRIVE_PATH}/agroguardai-lora-adapter/

In [ ]:
# @title 7. Evaluate against held-out test set
# Runs safety, dialect appropriateness, and hallucination checks
%cd /content/agroguardai-llm
!python src/evaluate.py \
    --base-model mistralai/Mistral-7B-Instruct-v0.3 \
    --adapter models/agroguardai-lora-adapter \
    --test-set data/processed/test.json \
    --output results/evaluation_report.json